In [18]:
%load_ext autoreload
%autoreload 2
import torch

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel


device = C.get_device()
print(f"Using device: {device}")

checkpoint_path = "../trained_models/ctc_all_augmentations_45epochs.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = CTCModel().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cpu


CTCModel(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (14):

In [19]:
from src.wordmaker import PHONEME_TO_LETTERS, levenshtein_distance, phonemes_to_text, WLIST1000

In [20]:
from src.ctc.features import wav_path_to_logmel
from src.ctc.dataset import textgrid_to_phone_ids
from src.ctc.metrics import greedy_decode, decode_to_phones, compute_per

wav_path = "../AutorskieDane/AutorskiDataset/id2.wav"
tg_path = "../AutorskieDane/AutorskiDataset/id2.TextGrid"
#wav_path = "../slowa_testowe/alibaba.wav"
#tf_path = "../slowa_testowe/"


target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)


mel = wav_path_to_logmel(wav_path)  # (F, T)
mel = mel.unsqueeze(0).to(device)  # (1, F, T)

with torch.no_grad():
    logits = model(mel)  # (1, T', C+1)

pred_ids = greedy_decode(logits.cpu())[0]  # list[int]


per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

PER for this utterance: 0.1907
Target:      sil s t u d e n tsj i c e r u n k u sil d o v a d u j oc5 sj e sil j a k sil p o s t e m p o v a tsj sil z d u Z i2 m i i l o sj tsj a m i d a n i2 h sil p o h o dz o n c i2 m i sil z r u Z n i2 h zj r u d e w sil u tS oc5 sj e r u v n~ e S j e i n t e r p r e t o v a tsj sil f S i2 s t k o t o sil v o p a r tsj u o sil p r o g r a m i2 sil i sil a l g o r i2 t m i2 k o m p u t e r o v e sil a sil t a k S e sil v e dz e sil s m a t e m a t i2 c i sil s t a t i2 s t i2 c i sil i e k o n o m j i sil
Prediction:  sil s t u d e n tsj k i e r u n k u sil p o v i e d u j oc5 tsj i a sil j a k sil p o s t a p o v a tsj sil z d u Z i2 m i i o sj tsj a m i n d o n i2 h sil p o v o dz oc5 c i2 m i sil z r u Z n i2 h zj r u d e w sil u tS oc5 sj i e r u v n i e S j e i n t e r p r e t o v a tsj sil v S i2 s t k o t o sil v o p a r tsj u w sil o p r o g r a m e sil i a l g o r i2 t v e k o m p u t e r o v e sil a t a g Z e m v i e dz oc5 z m o t e m a t 

In [21]:
from pathlib import Path

from src.ctc.features import wav_path_to_logmel
from src.ctc.dataset import textgrid_to_phone_ids
from src.ctc.metrics import greedy_decode, decode_to_phones, compute_per

dataset_dir = Path("../AutorskieDane/AutorskiDataset")

all_preds: list[list[int]] = []
all_targets: list[list[int]] = []
per_utterance: list[float] = []

for tg_path in sorted(dataset_dir.glob("*.TextGrid")):
    wav_path = tg_path.with_suffix(".wav")
    if not wav_path.exists():
        continue

    target_ids = textgrid_to_phone_ids(str(tg_path), map_sp_to_sil=True)

    mel = wav_path_to_logmel(str(wav_path)).unsqueeze(0).to(device)  # (1, F, T)
    with torch.no_grad():
        logits = model(mel)  # (1, T', C+1)
    pred_ids = greedy_decode(logits.cpu())[0]  # list[int]

    per = compute_per([pred_ids], [target_ids])
    per_utterance.append(per)
    all_preds.append(pred_ids)
    all_targets.append(target_ids)

    print(f"=== {tg_path.stem} | PER: {per:.4f} ===")
    print("Target:     ", decode_to_phones(target_ids))
    print("Prediction: ", decode_to_phones(pred_ids))
    print()

macro_per = sum(per_utterance) / len(per_utterance) if per_utterance else 0.0
micro_per = compute_per(all_preds, all_targets)

print("=" * 70)
print(f"Utterances evaluated: {len(per_utterance)}")
print(f"Average PER (macro, mean over utterances):        {macro_per:.4f}")
print(f"Overall PER (micro, total edits / total phones):  {micro_per:.4f}")


=== brutus | PER: 0.5214 ===
Target:      sil v i2 S e d w e m z d o m u sil m i n u s p e t n a sj tsj e n o c t u l i m n~ e m r u s t S i2 m a z a r e n k e p r o v a dzj i sil f s k a z u j e m i m u j c e l sil z w o t i2 sil s w o t c i sil t o f S i2 s t k o z a tS i2 n a sj e o t s k u r i2 p a l c e n a e r a m o n a n~ e b e s c e sil n~ e tS u j e t f a Z i2 sil i v k o n~ c u b r a k d o t i2 k u sil b o o t o j e s t m u j sil p a n k t u r i2 t S i2 m a m n~ e z o j c o f s k oc5 t r o s k oc5 sil n a s t e m p n~ e n~ i S tS i2 r o z dzj e r a i z a p a l a p w o m e n~ j a s n e j sj f e c i2 v m o i h sil f p r o v a dz a m n~ e v k o l e j n e k r e n i m u v o n c g dzj e m a m i sj tsj sil p a t S s i2 n u sil o t o t e s t o p n~ e sil tsj e n~ i Z e j n a S tS i2 t i2 v z n~ e sj e n~ a t f e g o sil t a k m u v i sil m i v p w u c a k r i2 S t a w c i l o d u sil t a k m u v i sil m o j e g a w c i sil i sil m o j oc5 m a s k e sil a j a z k a Z d i2 m k o l e j 

In [22]:
from src.levenshtein import lev_weighted
from src.levenshtein import damerau_lev
from src.levenshtein import true_damerau_levenshtein
from src.levenshtein import damerau_levenshtein_weighted
from src.levenshtein import levenshtein_phoneme_aware
from src.levenshtein import damerau_levenshtein_neighbour_aware


In [23]:
wav_path = "../slowa_testowe/wolowina.wav"
#wav_path = "../slowa_testowe/siedemnascie.wav"

target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)


mel = wav_path_to_logmel(wav_path)  # (F, T)
mel = mel.unsqueeze(0).to(device)  # (1, F, T)

with torch.no_grad():
    logits = model(mel)  # (1, T', C+1)

pred_ids = greedy_decode(logits.cpu())[0]  # list[int]


per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

decoded = decode_to_phones(pred_ids)
print("Decoded phonemes: ", decoded)
decoded_text = phonemes_to_text(decoded)
print("Decoded text (with silences): ", decoded_text)
#result = ''.join(p for p in decoded_text if p != 'sil')
result = decoded_text.replace("sil", "").replace(" ", "")
w = [str(ph) for ph in decoded_text.split()]
print("Decoded text (phonemes as strings): ", w)
wtext = phonemes_to_text(w, after_silence=True)
print("Decoded text (after removing silences): ", wtext)
output = WLIST1000[0]
mindist = 1000
for wr in WLIST1000:
    #print(f"Testing word: {wr}")
    #dist = levenshtein_distance(wr, w)
    dist = damerau_levenshtein_neighbour_aware(wr, w)
    #print(dist)
    if dist < mindist:
        mindist = dist
        output = wr
print("_______________________________________________________________")
print(f"Prediction without correction: {result}")
print(f"Best match:  ---- {output} ----- with distance {mindist}")

PER for this utterance: 0.8636
Target:      sil j e Z e l i t o sj e sil u d a sil t o s t f o Z i2 m i2 n a p r a v d e d u Z o d a n i2 h sil
Prediction:  sil p r o w o v i n a sil
Decoded phonemes:  sil p r o w o v i n a sil
Decoded text (with silences):  sil p r o ł o w i n a sil
Decoded text (phonemes as strings):  ['sil', 'p', 'r', 'o', 'ł', 'o', 'w', 'i', 'n', 'a', 'sil']
Decoded text (after removing silences):  prołołina
_______________________________________________________________
Prediction without correction: prołowina
Best match:  ---- wołowina ----- with distance 2.8000000000000003


In [24]:
wav_path = "../slowa_testowe/wolowina.wav"
wav_path = "../slowa_testowe/siedemnascie.wav"
target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)

mel = wav_path_to_logmel(wav_path)
mel = mel.unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(mel)

pred_ids = greedy_decode(logits.cpu())[0]

per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

# Fonemy z CTC
decoded = decode_to_phones(pred_ids)
print("Decoded phonemes:", decoded)

# JEDNA konwersja phonemes → tekst
decoded_text = phonemes_to_text(decoded, after_silence=False)
print("Decoded text:", decoded_text)

# Cleanup — usuń sil i spacje
result = decoded_text.replace("sil", "").replace(" ", "").strip()
print("Result:", result)

# Fuzzy match — używaj FONEMÓW, nie liter
output = WLIST1000[0]
mindist = 1000
for wr in WLIST1000:
    dist = damerau_levenshtein_neighbour_aware(wr, result)   # decoded = fonemy
    if dist < mindist:
        mindist = dist
        output = wr

print("_______________________________________________________________")
print(f"Prediction without correction: {result}")
print(f"Best match:  ---- {output} ----- with distance {mindist}")

PER for this utterance: 0.8409
Target:      sil j e Z e l i t o sj e sil u d a sil t o s t f o Z i2 m i2 n a p r a v d e d u Z o d a n i2 h sil
Prediction:  sil sj e m n o sj tsj sil
Decoded phonemes: sil sj e m n o sj tsj sil
Decoded text: sil sj e m n o sj tsj sil
Result: sjemnosjtsj
_______________________________________________________________
Prediction without correction: sjemnosjtsj
Best match:  ---- siostra ----- with distance 4.0


In [25]:
wav_path = "../slowa_testowe/wolowina.wav"
wav_path = "../slowa_testowe/siedemnascie.wav"
target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)

mel = wav_path_to_logmel(wav_path)
mel = mel.unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(mel)

pred_ids = greedy_decode(logits.cpu())[0]

per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

# NEW — od razu lista, nie string
decoded_phonemes = [C.IDX2LABEL.get(i, "<UNK>") for i in pred_ids]
print("Decoded phonemes:", decoded_phonemes)

# Konwersja do tekstu — JEDNA, na liście fonemów
decoded_text = phonemes_to_text(decoded_phonemes, after_silence=False)
print("Decoded text:", decoded_text)

# Cleanup (sil już znika w phonemes_to_text dzięki mappingowi 'sil':'')
result = decoded_text.replace(" ", "").strip()
print("Result:", result)

# Fuzzy match na fonemach (nie literach)
output = WLIST1000[0]
mindist = 1000
for wr in WLIST1000:
    dist = damerau_levenshtein_neighbour_aware(wr, result)
    if dist < mindist:
        mindist = dist
        output = wr

print("_______________________________________________________________")
print(f"Prediction without correction: {result}")
print(f"Best match: ---- {output} ----- with distance {mindist}")

PER for this utterance: 0.8409
Target:      sil j e Z e l i t o sj e sil u d a sil t o s t f o Z i2 m i2 n a p r a v d e d u Z o d a n i2 h sil
Prediction:  sil sj e m n o sj tsj sil
Decoded phonemes: ['sil', 'sj', 'e', 'm', 'n', 'o', 'sj', 'tsj', 'sil']
Decoded text: siemność
Result: siemność
_______________________________________________________________
Prediction without correction: siemność
Best match: ---- zimno ----- with distance 2.1


In [26]:
mel[:, :10].shape

torch.Size([1, 10, 182])

In [27]:
from src.wordmaker import dictionary_extend
null_dict = []
nd = dictionary_extend(null_dict, "../AutorskieDane/AutorskiDataset")


znaleziono 31 plików


In [28]:
from pathlib import Path
from src.wordmaker import parse_words

PATH = Path('../AutorskieDane/AutorskiDataset/Potop1.TextGrid')

with open(PATH, "r", encoding="utf-8") as f:
    wordss = parse_words(f.read())

wav_path = '../AutorskieDane/AutorskiDataset/Potop1.wav'

mel = wav_path_to_logmel(wav_path)

In [29]:
from src.levenshtein import lev_weighted
from src.levenshtein import damerau_lev
from src.levenshtein import true_damerau_levenshtein
from src.levenshtein import damerau_levenshtein_weighted
from src.levenshtein import levenshtein_phoneme_aware
from src.levenshtein import damerau_levenshtein_neighbour_aware
from src.wordmaker import folder_search_accuracy

In [30]:
def mel_cut(word_info, mel): #(start_time, end_time, word)
    hop_time = C.FRAME_MS / 1000
    n_start = int(word_info[0] / hop_time)
    n_end = int(word_info[1] / hop_time)
    mel_exact = mel[:,n_start:n_end]
    return mel_exact

def predict_from_exact_mel_ctc(target_word, mel_exact, dictionary, model, lev=lev_weighted, verbose=False):

    
    mel_exact = mel_exact.unsqueeze(0).to(device) 

    with torch.no_grad():
        logits = model(mel_exact)  # (1, T', C+1)
    
    pred_ids = greedy_decode(logits.cpu())[0]  # list[int]

    #decoded = decode_to_phones(pred_ids)
    #decoded_text = phonemes_to_text(decoded)
    decoded_phonemes = [C.IDX2LABEL.get(i, "<UNK>") for i in pred_ids]
    decoded_text = phonemes_to_text(decoded_phonemes, after_silence=False)
    #w = [str(ph) for ph in decoded_text.split()]
    result = decoded_text.replace(" ", "").strip()
    #wtext = phonemes_to_text(w, after_silence=False)
    
    output = dictionary[0]
    mindist = 1000
    for wr in dictionary:
        dist = lev(wr, result)
        if dist < mindist:
            mindist = dist
            output = wr
    if(verbose):
        print(f'output: {output}')
        print(f'target: {target_word}')
    return output

In [31]:
PATH = Path('../AutorskieDane/AutorskiDataset/lalka3.TextGrid') 
wav_path = '../AutorskieDane/AutorskiDataset/lalka3.wav'
#żeby se przetestować to usstaw tak aby w obu ścieżkach był ten sam plik 

with open(PATH, "r", encoding="utf-8") as f:
    wordss = parse_words(f.read())

mel = wav_path_to_logmel(wav_path)


good = 0 
for w_info in wordss:
    if(w_info[2] != 'sil' and (w_info[1] - w_info[0]) > C.WIN_MS/1000):
        #print(w_info[1] - w_info[0])
        #print(C.FRAME_MS/1000)
        mel_exact = mel_cut(w_info, mel)
        output = predict_from_exact_mel_ctc(w_info[2], mel_exact, null_dict, model)
        if output == w_info[2]:
            good += 1
print(good/len(wordss))

0.3877551020408163


In [32]:
from src.parsers import wav_to_logmel
def folder_search_accuracy_ctc(model, data_dir, dictionary, lev):
    correct = 0
    incorrect = 0 
    tg_paths = sorted(str(p) for p in Path(data_dir).rglob("*.TextGrid"))
    print(f"found {len(tg_paths)} files, search works")

    for tg in tg_paths:

        wav_path = tg[: -len(".TextGrid")] + ".wav"
        PATH = Path(tg)
        with open(PATH, "r", encoding="utf-8") as f:
            words_timeframes = parse_words(f.read())
        mel = wav_to_logmel(wav_path=wav_path)

        for w_info in words_timeframes:
            if(w_info[2] != 'sil' and (w_info[1] - w_info[0]) > C.WIN_MS/1000):
                mel_exact = mel_cut(w_info, mel)
                output = predict_from_exact_mel_ctc(w_info[2], mel_exact, dictionary, model, lev=lev, verbose=False)
                if output == w_info[2]:
                    correct += 1
                else:
                    incorrect += 1
                    
    return correct / (correct+incorrect)

In [33]:
datdir = '../AutorskieDane/AutorskiDataset/'

In [49]:
acc = folder_search_accuracy_ctc(model, datdir, null_dict, lev=levenshtein_phoneme_aware)
print(acc)

found 31 files, search works
0.39914163090128757


In [46]:
acc = folder_search_accuracy_ctc(model, datdir, null_dict, lev=lev_weighted)
print(acc)

found 31 files, search works


/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:42: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, audio = wavfile.read(wav_path)


0.3801348865726548


In [47]:
acc = folder_search_accuracy_ctc(model, datdir, null_dict, lev=damerau_lev)
print(acc)

found 31 files, search works


/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:42: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, audio = wavfile.read(wav_path)


0.3770692826486818


In [48]:
acc = folder_search_accuracy_ctc(model, datdir, null_dict, lev=true_damerau_levenshtein)
print(acc)

found 31 files, search works
0.3789086450030656


In [33]:
acc = folder_search_accuracy_ctc(model, datdir, null_dict, lev=damerau_levenshtein_neighbour_aware)
print(acc)

found 31 files, search works


C:\Users\mmapa\Desktop\Eti\ASR\ASR_project\src\parsers.py:42: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, audio = wavfile.read(wav_path)


0.5168608215818516


In [34]:
acc = folder_search_accuracy_ctc(model, datdir, null_dict, lev=damerau_levenshtein_neighbour_aware)
print(acc)

found 31 files, search works


C:\Users\mmapa\Desktop\Eti\ASR\ASR_project\src\parsers.py:42: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, audio = wavfile.read(wav_path)


0.5554874310239117


In [53]:
acc = folder_search_accuracy_ctc(model, datdir, null_dict, lev=damerau_levenshtein_weighted)
print(acc)

found 31 files, search works


/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:42: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, audio = wavfile.read(wav_path)


0.4696505211526671
